# 🎯 Customer Segmentation & Uplift Modeling

This notebook answers two critical business questions:
1. **Who are our customer segments?** (GMM clustering)
2. **Which customers will change behavior because of marketing?** (T-Learner uplift)

In [1]:
import sys
sys.path.insert(0, 'd:/ML_Project')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.config import *

import warnings
warnings.filterwarnings('ignore')

## 1. Customer Segments

A Gaussian Mixture Model (GMM) was fitted on 3 features: `log1p(CLV)`, `P(alive)`, and `recency_days`.
BIC model selection chose **k=7** as optimal.

In [2]:
segments = pd.read_parquet(OUTPUT_DATA_DIR / 'customer_segments.parquet')
print(f'Total customers segmented: {len(segments):,}')
print(f'Segments found: {segments["segment_name"].nunique()}')
segments['segment_name'].value_counts()

Total customers segmented: 4,244
Segments found: 4


segment_name
Promising      2565
Champions      1378
Hibernating     287
Lost             14
Name: count, dtype: int64

In [3]:
# Segment distribution
seg_counts = segments['segment_name'].value_counts().reset_index()
seg_counts.columns = ['Segment', 'Count']

fig = px.bar(seg_counts, x='Segment', y='Count',
             title='Customer Segment Distribution',
             color='Count', color_continuous_scale='Blues')
fig.show()

## 2. Segment Profiles

In [4]:
# Compute segment profiles
profiles = segments.groupby('segment_name').agg(
    count=('predicted_clv', 'size'),
    avg_clv=('predicted_clv', 'mean'),
    avg_p_alive=('p_alive', 'mean'),
).round(2).sort_values('avg_clv', ascending=False)

profiles['avg_clv'] = profiles['avg_clv'].map(lambda x: f'INR {x:,.0f}')
profiles

,count,avg_clv,avg_p_alive
segment_name,,,
Champions,1378,"INR 355,935",0.99
Promising,2565,"INR 64,626",0.98
Hibernating,287,"INR 36,640",0.76
Lost,14,"INR -46,913",0.86


In [5]:
# Interactive scatter: CLV vs P(alive) colored by segment
plot_df = segments[segments['predicted_clv'] < segments['predicted_clv'].quantile(0.95)].copy()

fig = px.scatter(plot_df, x='p_alive', y='predicted_clv',
                 color='segment_name', opacity=0.5,
                 title='CLV vs P(Alive) by Segment',
                 labels={'predicted_clv': 'Predicted CLV (INR)', 'p_alive': 'P(Alive)'})
fig.update_layout(legend_title='Segment', width=900, height=600)
fig.show()

## 3. Uplift Modeling (T-Learner)

**The key business insight**: Not all high-CLV customers should receive marketing.

- **Sure Things**: Will buy regardless of marketing (don't waste budget)
- **Persuadables**: Will buy *because* of marketing (target these!)
- **Lost Causes**: Won't buy regardless (don't waste budget)
- **Sleeping Dogs**: Will buy *less* if marketed to (definitely avoid)

> *Note: Treatment effects are simulated with a 15% uplift for the treated group, as this is observational data without a real A/B test.*

In [6]:
uplift = pd.read_parquet(OUTPUT_DATA_DIR / 'uplift_scores.parquet')
print(f'Total customers scored: {len(uplift):,}')
uplift['segment'].value_counts()

Total customers scored: 4,244


segment
Neutral/Negative    2971
Favourable           849
Persuadable          424
Name: count, dtype: int64

In [7]:
fig = px.histogram(uplift, x='uplift_score', nbins=50, color='segment',
                   title='Uplift Score Distribution by Segment',
                   labels={'uplift_score': 'Predicted Incremental Revenue (INR)'})
fig.show()

In [8]:
# Top persuadable customers
persuadable = uplift[uplift['segment'] == 'Persuadable'].nlargest(10, 'uplift_score')
persuadable['uplift_score'] = persuadable['uplift_score'].map(lambda x: f'INR {x:,.0f}')
persuadable

,customer_id,uplift_score,uplift_rank,segment
0,14911,"INR 9,973,821",1,Persuadable
1,17511,"INR 4,278,098",2,Persuadable
2,16754,"INR 3,328,372",3,Persuadable
3,16029,"INR 3,319,845",4,Persuadable
4,13687,"INR 2,822,963",5,Persuadable
5,13902,"INR 2,820,106",6,Persuadable
6,15769,"INR 2,718,864",7,Persuadable
7,14298,"INR 2,647,233",8,Persuadable
8,13081,"INR 2,452,331",9,Persuadable
9,17920,"INR 2,203,414",10,Persuadable


## 4. Evaluating Uplift: The Qini Curve

To prove our uplift model actually ranks persuadable customers better than random targeting, we plot the **Qini Curve**.
Since our dataset does not contain a real A/B test treatment flag, we use the simulated treatment and outcomes to evaluate the ranking.

In [9]:
from src.models.uplift import UpliftModel

# Load original features to re-simulate the exact treatment assignment used in training
features = pd.read_parquet(PROCESSED_DATA_DIR / 'customer_features.parquet')
holdout = pd.read_parquet(OUTPUT_DATA_DIR / 'holdout_actuals.parquet')
valid_idx = uplift['customer_id'].astype(int)
X = features.loc[valid_idx]
y = holdout['holdout_revenue'].reindex(valid_idx, fill_value=0)

uplift_model = UpliftModel()
X_sim, treatment, y_sim = uplift_model.simulate_treatment(X, y, random_state=42)

fig = uplift_model.plot_qini_curve(y_true=y_sim, uplift_scores=uplift['uplift_score'], treatment=treatment)
fig.show()

## Business Recommendations

| Segment | Action | Budget Priority |
|---------|--------|----------------|
| Champions | Loyalty rewards, VIP perks | Medium |
| At-Risk High-Value | Urgent win-back campaign | **HIGH** |
| Promising | Upsell & cross-sell | Medium |
| Hibernating | Re-engagement email | Low |
| Lost | Low-cost reactivation or suppress | Minimal |

**Key**: Focus the INR 50L quarterly budget on the **Persuadable** customers, not just the highest-CLV ones.